In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# Importing dataset
dataset = pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [3]:
# Handling categorical data
dataset = pd.get_dummies(dataset,drop_first=True,dtype=int)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [4]:
# Dropping the 'USER ID' Columns as it is unique identifier
dataset = dataset.drop("User ID",axis=1)
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [5]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [6]:
dataset['Purchased'].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [7]:
# Feature and target selection
independent = dataset[['Age', 'EstimatedSalary', 'Gender_Male']]
dependent = dataset['Purchased']

In [8]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [9]:
dependent

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [10]:
# Splitting data into training and test sets
x_train,x_test,y_train,y_test = train_test_split(independent,dependent,test_size=0.3,random_state=0)

In [11]:
x_train.shape

(280, 3)

In [12]:
x_test.shape

(120, 3)

In [13]:
y_train.shape

(280,)

In [14]:
y_test.shape

(120,)

In [15]:
# Data Standardization
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [16]:
# GridSearch CV
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Defining the different parameters for gridsearch optimization
param_grid = {'C':[0.1,1,10,100,1000],
              'gamma':['scale','auto'],
              'kernel':['rbf','sigmoid','poly','linear']}


grid = GridSearchCV(SVC(probability=True),param_grid,refit=True,verbose=3,scoring='f1_weighted',n_jobs=-1)
grid.fit(x_train,y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


GridSearchCV(estimator=SVC(probability=True), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['rbf', 'sigmoid', 'poly', 'linear']},
             scoring='f1_weighted', verbose=3)

In [17]:
# Best gridsearch parameter
print(grid.best_params_)

{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}


In [18]:
# Grid prediction
y_pred = grid.predict(x_test)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 1, 1])

In [19]:
# evaluation metrics
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[73  6]
 [ 4 37]]


In [20]:
from sklearn.metrics import classification_report
report = classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.95      0.92      0.94        79
           1       0.86      0.90      0.88        41

    accuracy                           0.92       120
   macro avg       0.90      0.91      0.91       120
weighted avg       0.92      0.92      0.92       120



In [21]:
from sklearn.metrics import f1_score
f1_weighted= f1_score(y_test,y_pred,average='weighted')
print(f1_weighted)

0.9171245421245421


In [22]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])

np.float64(0.9654214263661625)

In [23]:
results = grid.cv_results_
table = pd.DataFrame.from_dict(results)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.015582,0.001549,0.006660,0.000613,0.1,scale,rbf,"{'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}",0.855314,0.857143,0.821429,0.964572,0.963889,0.892469,0.059958,7
1,0.012966,0.002432,0.005699,0.000789,0.1,scale,sigmoid,"{'C': 0.1, 'gamma': 'scale', 'kernel': 'sigmoid'}",0.753521,0.721707,0.644599,0.909115,0.847662,0.775321,0.093351,29
2,0.007315,0.001171,0.004195,0.000932,0.1,scale,poly,"{'C': 0.1, 'gamma': 'scale', 'kernel': 'poly'}",0.755102,0.747056,0.618196,0.890114,0.758537,0.753801,0.086069,32
3,0.006375,0.001008,0.004109,0.000683,0.1,scale,linear,"{'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}",0.812540,0.763089,0.644599,0.909115,0.888158,0.803500,0.095169,27
4,0.012409,0.001332,0.004488,0.000584,0.1,auto,rbf,"{'C': 0.1, 'gamma': 'auto', 'kernel': 'rbf'}",0.855314,0.857143,0.821429,0.964572,0.963889,0.892469,0.059958,7
5,0.008111,0.000892,0.003718,0.000618,0.1,auto,sigmoid,"{'C': 0.1, 'gamma': 'auto', 'kernel': 'sigmoid'}",0.753521,0.721707,0.644599,0.909115,0.847662,0.775321,0.093351,29
6,0.005651,0.000751,0.004192,0.000949,0.1,auto,poly,"{'C': 0.1, 'gamma': 'auto', 'kernel': 'poly'}",0.802555,0.723577,0.618196,0.870721,0.758537,0.754717,0.084104,31
7,0.005750,0.001017,0.003968,0.002203,0.1,auto,linear,"{'C': 0.1, 'gamma': 'auto', 'kernel': 'linear'}",0.812540,0.763089,0.644599,0.909115,0.888158,0.803500,0.095169,27
8,0.006887,0.000527,0.004118,0.000775,1.0,scale,rbf,"{'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}",0.874254,0.875644,0.823129,0.947015,0.964286,0.896866,0.051883,3
9,0.006827,0.001007,0.004246,0.001892,1.0,scale,sigmoid,"{'C': 1, 'gamma': 'scale', 'kernel': 'sigmoid'}",0.726641,0.715601,0.601841,0.806465,0.800053,0.730120,0.074024,39


In [24]:
age = int(input("Enter your age:"))
salary = float(input("Enter your salary"))
gender = int(input("male=1,female=0"))

In [25]:
data = sc.transform([[age,salary,gender]])

C:\Users\thiru\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [26]:
result = grid.predict(data)
result

array([0])